In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
cinic_dir = "./cinic10" 

In [4]:
cinic_mean = [0.47889522, 0.47227842, 0.43047404]
cinic_std = [0.24205776, 0.23828046, 0.25874835]

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=cinic_mean, std=cinic_std)
])

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cinic_mean, std=cinic_std)
])


In [5]:
train_dataset = torchvision.datasets.ImageFolder(
    cinic_dir + "/train",
    transform=train_transform
)

valid_dataset = torchvision.datasets.ImageFolder(
    cinic_dir + "/valid",
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=8,
    pin_memory=True,
    persistent_workers=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False, num_workers=8,
    pin_memory=True,
    persistent_workers=True)

In [ ]:
import torch
import torch.nn as nn
class CIFAR10CNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2), 

  
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  


            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)  
        )

        self.pool = nn.AdaptiveAvgPool2d(1)  

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

class CNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


model = CIFAR10CNN().to(device)

In [7]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)
# optimizer = torch.optim.SGD(
#     model.parameters(),
#     lr=0.1,
#     momentum=0.9,
#     weight_decay=5e-4
# )

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5   
)


In [8]:
class EarlyStopping:

    def __init__(self, patience=5, path="best_model.pth"):

        self.patience = patience
        self.path = path

        self.best_loss = float("inf")
        self.counter = 0
        self.stop = False

    def __call__(self, val_loss, model):

        if val_loss < self.best_loss:

            self.best_loss = val_loss
            self.counter = 0
    
            torch.save(model.state_dict(), self.path)

        else:
            self.counter += 1

            if self.counter >= self.patience:
                self.stop = True

In [9]:
def validate(model, loader):

    model.eval()
    total_loss = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
early_stopping = EarlyStopping(patience=30)

epochs = 100

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)


        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    val_loss = validate(model, valid_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")

    early_stopping(val_loss, model)

    if early_stopping.stop:
        break

    scheduler.step()


Epoch 1
Train Loss: 1.5522
Val Loss: 1.4302

Epoch 2
Train Loss: 1.2409
Val Loss: 1.1624

Epoch 3
Train Loss: 1.1037
Val Loss: 1.0817

Epoch 4
Train Loss: 1.0076
Val Loss: 0.9564

Epoch 5
Train Loss: 0.9317
Val Loss: 0.9729

Epoch 6
Train Loss: 0.8294
Val Loss: 0.7838

Epoch 7
Train Loss: 0.7929
Val Loss: 0.7864

Epoch 8
Train Loss: 0.7658
Val Loss: 0.7558

Epoch 9
Train Loss: 0.7386
Val Loss: 0.7356

Epoch 10
Train Loss: 0.7205
Val Loss: 0.7323

Epoch 11
Train Loss: 0.6670
Val Loss: 0.6864

Epoch 12
Train Loss: 0.6464
Val Loss: 0.6735

Epoch 13
Train Loss: 0.6387
Val Loss: 0.6592

Epoch 14
Train Loss: 0.6199
Val Loss: 0.6732

Epoch 15
Train Loss: 0.6138
Val Loss: 0.6636

Epoch 16
Train Loss: 0.5833
Val Loss: 0.6335

Epoch 17
Train Loss: 0.5707
Val Loss: 0.6172

Epoch 18
Train Loss: 0.5677
Val Loss: 0.6258

Epoch 19
Train Loss: 0.5615
Val Loss: 0.6368

Epoch 20
Train Loss: 0.5574
Val Loss: 0.6214

Epoch 21
Train Loss: 0.5391
Val Loss: 0.6112

Epoch 22
Train Loss: 0.5366
Val Loss: 0.61

In [12]:
best_model = CIFAR10CNN().to(device)
best_model.load_state_dict(torch.load("best_model.pth"))
best_model.to(device)
best_model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in valid_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = best_model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Valid Accuracy:", 100 * correct / total)

Valid Accuracy: 79.11777777777777
